# J07 — Infiltration

**Physique et hydrodynamique des sols** — S. J. Gumiere

*Atelier du Jour 7 : Green–Ampt et Mein–Larson, ajustement Philip/Horton, sols stratifiés et croûte de battance dans HYDRUS-1D*

> Version **étudiant** : complétez les cellules marquées `# À COMPLÉTER`.

## Mise en place

Les fonctions de Mualem–van Genuchten du Jour 5 sont fournies. Conventions HYDRUS-1D : $z$ positif vers le haut, flux $q > 0$ vers le haut
(`vTop` < 0 = infiltration), profondeurs négatives. Unités : cm et jours (les résultats sont convertis en mm/h quand c'est parlant).

In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize, integrate
from hydrus_io import read_tlevel, read_nod_inf, read_obs_node, read_balance

plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True, "grid.alpha": 0.3})
HYD = "../../hydrus"
SOLS = {
    "sable":         [0.045, 0.430, 0.145, 2.68, 712.8, 0.5],
    "loam sableux":  [0.065, 0.410, 0.075, 1.89, 106.1, 0.5],
    "loam":          [0.078, 0.430, 0.036, 1.56, 24.96, 0.5],
    "loam limoneux": [0.067, 0.450, 0.020, 1.41, 10.80, 0.5],
}

def vg_theta(h, p):
    thr, ths, a, n, Ks, l = p; m = 1 - 1 / n
    h = np.asarray(h, float)
    Se = np.where(h < 0, (1 + (a * np.abs(h))**n)**(-m), 1.0)
    return thr + (ths - thr) * Se

def vg_K(h, p):
    thr, ths, a, n, Ks, l = p; m = 1 - 1 / n
    h = np.asarray(h, float)
    Se = np.where(h < 0, (1 + (a * np.abs(h))**n)**(-m), 1.0)
    return Ks * Se**l * (1 - (1 - Se**(1 / m))**m)**2

def h_front(p, h_i=-1e4):
    """Succion au front de Green–Ampt |h_f| = int_{h_i}^{0} K_r(h) dh (cm)."""
    return integrate.quad(lambda h: vg_K(h, p) / p[4], h_i, 0, limit=400, points=[-1000, -100, -10, -1])[0]

## Exercice 1 — Green–Ampt sous lame d'eau et sous pluie (Mein–Larson)  *(≈ 20 min)*

1. Écrire `green_ampt_cumul(t, Ks, hf, dtheta, h0=0)` : infiltration cumulée sous lame d'eau $h_0$, solution de
   $I - B\ln(1 + I/B) = K_s t$ avec $B = (h_0 + |h_f|)\Delta\theta$ (`optimize.brentq`), et `green_ampt_taux(I, ...)` $= K_s(1 + B/I)$.
2. Écrire `temps_submersion(i, Ks, hf, dtheta)` (Mein & Larson) : $I_p = K_s|h_f|\Delta\theta/(i - K_s)$, $t_p = I_p/i$,
   et `green_ampt_pluie(t, i, ...)` : $I = it$ pour $t \le t_p$, puis $I - I_p - B\ln[(B + I)/(B + I_p)] = K_s(t - t_p)$.
3. Loam de Carsel & Parrish, $h_i = -300$ cm : calculer $\Delta\theta = \theta_s - \theta(h_i)$ et $|h_f| = \int K_r\,dh$ (`h_front`) ;
   tracer $I(t)$ et $i(t)$ (a) sous lame d'eau nulle avec $h_i = -100$ cm, comparé à `sum(Infil)` et `vTop` de
   `J06_infiltration_submergee_loam` (0–0,6 j) ; (b) sous pluie de 30 cm/j, comparé à `J06_pluie_flux_impose_loam`.
4. Déterminer le $|h_f|$ « effectif » qui reproduit le temps de submersion HYDRUS ($t_p \approx 0{,}088$ j) et commenter.

In [ ]:
def green_ampt_cumul(t, Ks, hf, dtheta, h0=0.0):
    """I(t) (cm) sous lame d'eau h0 : résolution implicite de I - B ln(1 + I/B) = Ks t."""
    B = (hf + h0) * dtheta
    t = np.atleast_1d(np.asarray(t, float))
    # À COMPLÉTER : pour chaque t, brentq(lambda I: I - B*np.log(1 + I/B) - Ks*t, 1e-9, Ks*t + 50*B + 1)
    pass

def green_ampt_taux(I, Ks, hf, dtheta, h0=0.0):
    return # À COMPLÉTER

def temps_submersion(i, Ks, hf, dtheta):
    # À COMPLÉTER : retourne (t_p, I_p)
    pass

def green_ampt_pluie(t, i, Ks, hf, dtheta):
    """I(t) sous pluie constante i > Ks (Mein & Larson 1973)."""
    # À COMPLÉTER
    pass

p = SOLS["loam"]; Ks = p[4]
hf = h_front(p); dth300 = p[1] - float(vg_theta(-300, p)); dth100 = p[1] - float(vg_theta(-100, p))
print(hf, dth300, dth100)

## Exercice 2 — Essai au double anneau : Philip et Horton  *(≈ 15 min)*

`data/J07_double_anneau.csv` : infiltration cumulée $I$ (cm) mesurée pendant 3 h sur deux sites (A : loam ; B : loam sableux structuré).

1. Ajuster par `optimize.curve_fit` le modèle de Philip $I = S\sqrt{t} + At$ et celui de Horton
   $I = i_f t + (i_0 - i_f)(1 - e^{-kt})/k$ (temps en h) ; donner les paramètres et la RMSE de chaque modèle et de chaque site.
2. Tracer $I(t)$ mesuré et ajusté, puis $i(t)$ des deux modèles jusqu'à 6 h (extrapolation) ; comparer $A$ et $i_f$.
3. Estimer $K_s$ ($\approx A/0{,}4$ et $\approx i_f$), le temps gravitaire $t_{grav} = (S/K_s)^2$ et la longueur capillaire
   $\lambda_c = b S^2/(\Delta\theta K_s)$ (White & Sully, $b = 0{,}55$, $\Delta\theta = 0{,}25$) ; comparer aux valeurs de Carsel & Parrish.

In [ ]:
da = pd.read_csv("data/J07_double_anneau.csv")
philip = lambda t, S, A: S * np.sqrt(t) + A * t
horton = lambda t, i0, i_f, k: i_f * t + (i0 - i_f) * (1 - np.exp(-k * t)) / k
for site, g in da.groupby("site"):
    t = g["t_min"].to_numpy() / 60.0; I = g["I_cm"].to_numpy()
    # À COMPLÉTER : pP, _ = optimize.curve_fit(philip, t, I, p0=[1, 1]) ; pH, _ = optimize.curve_fit(horton, t, I, p0=[5, 1, 2], bounds=(0, [50, 20, 50]))
    pass

## Exercice 3 — Sols stratifiés dans HYDRUS-1D : loam sur sable, sable sur loam  *(≈ 15 min)*

Projets `J07_stratifie_loam_sur_sable` et `J07_stratifie_sable_sur_loam` : deux couches (0–30 cm / 30–100 cm), $h_i = -200$ cm,
lame d'eau nulle en surface, drainage libre, 1 j, nœuds d'observation à 10, 25, 35 et 60 cm.

1. Tracer les profils $\theta(z)$ et $h(z)$ (`read_nod_inf`) à 0,1 ; 0,25 ; 0,5 ; 1 j pour les deux projets.
2. Tracer `vTop`$(t)$ et `sum(Infil)`$(t)$ (`read_tlevel`) ; donner $I(1\,\mathrm{j})$ (attendu : 26,1 cm contre 45,1 cm) et le flux
   final au bas.
3. Position du front en fonction du temps (premier nœud, en partant de la surface, où $\theta < \theta_i + 0{,}02$) pour chaque temps
   d'impression ; identifier l'arrêt à l'interface.
4. `read_obs_node` : $h(t)$ à 25 et 35 cm ; vérifier, pour loam sur sable, que le sable ne transmet le flux qu'une fois $h$ à l'interface
   remonté à $\approx -9$ cm (résoudre $K_{sable}(h) = K_{s,loam}$ avec `brentq`) et que $\theta_{sable}$ vaut alors $\approx 0{,}235$.

In [ ]:
projets = {"loam sur sable": f"{HYD}/J07_stratifie_loam_sur_sable", "sable sur loam": f"{HYD}/J07_stratifie_sable_sur_loam"}
# 1. profils : nod = read_nod_inf(P) ; nod[t]["Moisture"], nod[t]["Head"], nod[t]["Depth"]
# 2. tl = read_tlevel(P) : tl["vTop"], tl["sum(Infil)"], tl["vBot"]
# 3. front : pour chaque t, z du premier nœud où theta < theta_i + 0.02
# 4. ob = read_obs_node(P) : ob[(26, "h")], ob[(36, "h")] ; h* tel que vg_K(h*, SOLS["sable"]) = SOLS["loam"][4]
# À COMPLÉTER

## Exercice 4 — Croûte de battance dans HYDRUS-1D  *(≈ 10 min)*

Projets `J07_croute_sans_croute` (loam, 101 nœuds) et `J07_croute_avec_croute` (croûte de 1 cm à $K_s = 0{,}5$ cm/j, 201 nœuds) :
pluie de 10 cm/j pendant 0,5 j, $h_i = -300$ cm, condition atmosphérique avec ruissellement.

1. Tracer `vTop`, `sum(Infil)`, `sum(RunOff)` et `hTop` pour les deux projets ; temps de submersion, infiltration et ruissellement
   totaux, coefficient de ruissellement.
2. Taux d'infiltration stationnaire sous lame d'eau nulle avec croûte ($K_s$ « effective » du système) ; le comparer à $K_s$ du loam.
3. Profils $h(z)$ sur 0–20 cm à 0,25 j (`read_nod_inf`) : pression $h_{sub}$ sous la croûte, gradient de charge à travers la croûte,
   résistance hydraulique $R_c = \Delta H / i$ ; vérifier que $K_{loam}(h_{sub}) < i$ (le sol sous la croûte n'est pas à gradient unitaire).

In [ ]:
crout = {"sans croûte": f"{HYD}/J07_croute_sans_croute", "avec croûte": f"{HYD}/J07_croute_avec_croute"}
# 1. tl = read_tlevel(P) : tl["vTop"], tl["sum(Infil)"], tl["sum(RunOff)"], tl["hTop"], tl["RunOff"]
# 2. i_stat = -tl["vTop"][(tl.index > 0.2) & (tl.index < 0.5)].mean()
# 3. nod = read_nod_inf(P) ; nod[0.25] : Head, Depth
# À COMPLÉTER

## Exercice 5 — Bonus — Green–Ampt à deux couches (sable sur loam)  *(≈ facultatif)*

Généraliser Green–Ampt à un profil à deux couches (Childs & Bybordi 1969) : quand le front est dans la couche 2 (profondeur $L_f > L_1$),
la conductivité effective en série est $K_{eff} = L_f/[L_1/K_1 + (L_f - L_1)/K_2]$ et $i = K_{eff}(h_0 + |h_{f,2}| + L_f)/L_f$,
avec $dL_f/dt = i/\Delta\theta_2$. Intégrer (`integrate.solve_ivp`) pour sable (30 cm) sur loam, $h_i = -200$ cm, lame nulle, et comparer
$I(t)$ et $i(t)$ à `J07_stratifie_sable_sur_loam` (45,1 cm à 1 j). Pourquoi le cas loam sur sable ne se traite-t-il pas ainsi ?

In [ ]:
p1, p2 = SOLS["sable"], SOLS["loam"]; L1 = 30.0
dth1 = p1[1] - float(vg_theta(-200, p1)); dth2 = p2[1] - float(vg_theta(-200, p2))
hf1, hf2 = h_front(p1, -200), h_front(p2, -200)
# À COMPLÉTER : dL/dt = i(L)/dtheta(L) avec i(L) = Keff(L) (hf(L) + L)/L ; solve_ivp de L = 1e-3 à t = 1 j

## Pour aller plus loin

* Avec le solveur `richards_1d` du Jour 6 (paramètres par nœud), reproduire `J07_stratifie_loam_sur_sable` et tester l'effet de $\Delta z$ à l'interface.
* Ajuster Green–Ampt ($K_s$, $h_f$) sur les courbes du double anneau et comparer $S = \sqrt{2 K_s |h_f| \Delta\theta}$ au $S$ de Philip.
* Calculer la lame ruisselée SCS-CN d'une pluie de 60 mm pour $CN$ = 60, 78, 90 et la comparer au ruissellement Green–Ampt (ex. 1) pour la même pluie en 1 h, 3 h et 12 h.